## Réseau cristallin du $\text{NaNO}_2$
STIEVENARD Alexandre - 79822300  
NOM Prénom - XXXXXXXX  
NOM Prénom - XXXXXXXX  
BODART Quentin - 70362100

#### 1. Imports, téléchargement de la structure et fonctions utiles

In [96]:
### IMPORTS ###
try:
    import numpy as np
    from numpy.linalg import norm
    import py3Dmol
    from mp_api.client import MPRester
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer,Structure
    import pandas as pd

except ImportError as e:
    print(f"module or submodule \"{e.name}\" not found.\
        \nPlease run the following command in the same folder as this notebook to ensure all needed modules are installed:\
        \npip install -r requirements.txt")
    raise SystemExit

In [97]:
### TELECHARGEMENT
mp_key = "cLLBrCi8gU7SqhnPKS4NJqSKB4Y6d0gD"
mp_id = "mp-2964"

with MPRester(mp_key) as mpr: structure = mpr.get_structure_by_material_id(mp_id)
sga = SpacegroupAnalyzer(structure)

Retrieving MaterialsDoc documents: 100%|██████████| 1/1 [00:00<?, ?it/s]


In [98]:
### FONCTIONS ###
def print_title(title:str):
    print('\n' + title + '\n' + "-"*len(title))

def get_angle(v1:np.ndarray, v2:np.ndarray) -> float:
    """Retourne l'angle en degrés entre deux vecteurs 3D"""
    return np.rad2deg(np.arccos(np.dot(v1,v2) / (norm(v1)*norm(v2))))

def py3Dmol_visualise(structure:Structure):
    """Crée une fenêtre de visualisation interactive de la structure donnée"""
    view = py3Dmol.view(
        width=700, 
        height=500,
        data=structure.to(fmt="cif"),
        format="cif",
        style={"sphere": {"scale": 0.3}, "stick": {"radius": 0.1}}
    )
    view.addUnitCell()
    view.zoomTo()
    view.show()

#### 2. Vecteurs de bases, type de maille, système cristallin et  groupe ponctuel

##### 2.1 Visualisation 

2.1.1. Maille primitive

In [99]:
py3Dmol_visualise(structure)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

2.1.2. Maille conventionelle

In [100]:
py3Dmol_visualise(structure.to_conventional())

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

##### 2.2 Vecteurs de bases 

In [ ]:
### FONCTIONS DE VISUALISATION ###
def lattice_vectors_df(matrix: np.ndarray) -> pd.DataFrame:
    """
    matrix : tableau de taille (3,3) où les lignes correspondent aux vecteurs a, b, c en coordonnées cartésiennes.
    Retourne un DataFrame contenant les composantes et les normes.
    """
    labels = ["a", "b", "c"]
    rows = []
    for i, lab in enumerate(labels):
        vec = matrix[i]
        rows.append({
            "vecteur": lab,
            "x": float(vec[0]),
            "y": float(vec[1]),
            "z": float(vec[2]),
            "norme": float(norm(vec)),
        })
    df = pd.DataFrame(rows, columns=["vecteur", "x", "y", "z", "norme"])
    return df

def lattice_angles_df(matrix: np.ndarray) -> pd.DataFrame:
    """
    Calcule alpha = angle(b,c), beta = angle(c,a), gamma = angle(a,b) en degrés
    en utilisant la fonction existante get_angle(v1, v2).
    """
    alpha = get_angle(matrix[1], matrix[2])
    beta  = get_angle(matrix[2], matrix[0])
    gamma = get_angle(matrix[0], matrix[1])
    return pd.DataFrame({
        "angle": ["α", "β", "γ"],
        "valeur (°)": [alpha, beta, gamma]
    })

def show_lattice(title: str, matrix: np.ndarray, precision: int = 2) -> None:
    """
    Affiche un titre puis les DataFrames des vecteurs du réseau (avec leurs normes)
    et des angles associés, avec un arrondi à la précision donnée.
    """
    print_title(title)
    df_vec = lattice_vectors_df(matrix).round(precision)
    df_ang = lattice_angles_df(matrix).round(precision)

    display(df_vec.round(precision))
    display(df_ang.round(precision))


In [102]:

### RESEAU DIRECT PRIMITIF ###
matrix = np.array(structure.lattice.matrix)
show_lattice("Vecteurs de base du réseau direct de la maille primitive (en Å)", matrix)

### RESEAU RECIPROQUE PRIMITIF ###

recp_matrix = np.array(structure.lattice.reciprocal_lattice.matrix)
show_lattice("Vecteurs de base du réseau réciproque de la maille primitive (en Å⁻¹)", recp_matrix)


Vecteurs de base du réseau direct de la maille primitive (en Å)
---------------------------------------------------------------


,x,y,z,norme
vecteur,,,,
a,3.17,-0.00,1.44,3.48
b,1.55,3.83,0.80,4.20
c,-0.00,-0.01,4.20,4.20


,angle,valeur (°)
0,α,79.16
1,β,65.56
2,γ,65.56



Vecteurs de base du réseau réciproque de la maille primitive (en Å⁻¹)
---------------------------------------------------------------------


,x,y,z,norme
vecteur,,,,
a,1.98,-0.80,-0.00,2.14
b,-0.00,1.64,0.00,1.64
c,-0.68,-0.04,1.49,1.64


,angle,valeur (°)
0,α,91.17
1,β,112.06
2,γ,112.06


In [103]:
### RESEAU DIRECT CONVENTIONNEL ###
matrix = np.array(structure.to_conventional().lattice.matrix)
show_lattice("Vecteurs de base du réseau direct de la maille conventionelle (en Å)", matrix)

### RESEAU RECIPROQUE CONVENTIONNEL ###

recp_matrix = np.array(structure.to_conventional().lattice.reciprocal_lattice.matrix)
show_lattice("Vecteurs de base du réseau réciproque de la maille conventionnelle (en Å⁻¹)", recp_matrix)


Vecteurs de base du réseau direct de la maille conventionelle (en Å)
--------------------------------------------------------------------


,x,y,z,norme
vecteur,,,,
a,3.48,0.00,0.00,3.48
b,0.00,5.36,0.00,5.36
c,0.00,0.00,5.47,5.47


,angle,valeur (°)
0,α,90.0
1,β,90.0
2,γ,90.0



Vecteurs de base du réseau réciproque de la maille conventionnelle (en Å⁻¹)
---------------------------------------------------------------------------


,x,y,z,norme
vecteur,,,,
a,1.81,-0.00,0.00,1.81
b,0.00,1.17,0.00,1.17
c,-0.00,-0.00,1.15,1.15


,angle,valeur (°)
0,α,90.0
1,β,90.0
2,γ,90.0


In [104]:
group_symbol_to_french = {
    "P" : "Primitive",
    "I" : "Centrée",
    "A" : "Bases centrées (A)",
    "B" : "Bases centrées (B)",
    "C" : "Bases centrées (C)",
    "F" : "Faces centrées"
}
crystal_system_to_french = {
    "cubic" : "Cubique",
    "hexagonal" : "Hexagonal",
    "monoclinic" : "Monoclinique",
    "orthorhombic" : "Orthorombique",
    "tetragonal" : "Tétragonal",
    "triclinic" : "Triclinique",
    "trigonal" : "Trigonal"
}

### TYPE DE MAILLE ###
print_title("Type de maille")
print(group_symbol_to_french[sga.get_space_group_symbol()[0]])

### SYSTEME CRYSTALLIN ###
print_title("Système cristallin")
print(crystal_system_to_french[sga.get_crystal_system()])

### GROUPE PONCTUEL ###
print_title("Groupe ponctuel")
print(sga.get_point_group_symbol())


Type de maille
--------------
Centrée

Système cristallin
------------------
Orthorombique

Groupe ponctuel
---------------
mm2
